# 02 - Data Profiling

## Objective

This notebook examines the structure and quality of the raw flight dataset before exploratory analysis and data cleaning.

The profiling process includes:

- Reviewing dataset dimensions and schema
- Measuring missing values
- Identifying duplicate records
- Summarizing numerical variables
- Reviewing key categorical variables
- Performing essential data-quality checks

The findings from this notebook will guide the exploratory analysis and data-cleaning decisions.

#### Load configuration and dataset

In [0]:
# Load the project configuration and raw flight table

from config import project_config as cfg
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

df_raw = spark.read.table(cfg.RAW_TABLE)

print(f"Table loaded: {cfg.RAW_TABLE}")

Table loaded: workspace.default.flights_raw


#### Dataset overview

This section reviews the size and structure of the raw flight dataset.

In [0]:
# Review dataset dimensions

row_count = df_raw.count()
column_count = len(df_raw.columns)

print(f"Total records: {row_count:,}")
print(f"Total columns: {column_count}")

Total records: 7,001,619
Total columns: 32


In [0]:
# Review the inferred schema

df_raw.printSchema()

root
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_NM: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DEL15: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELA

#### Missing-value analysis

Missing values are measured for every column to identify variables that may require removal, imputation, or special treatment during data cleaning.

In [0]:
# Calculate missing-value counts for every column

null_counts = (
    df_raw
    .select(
        [
            F.sum(
                F.col(column_name).isNull().cast("long")
            ).alias(column_name)
            for column_name in df_raw.columns
        ]
    )
    .first()
)

missing_values_data = [
    (
        field.name,
        field.dataType.simpleString(),
        int(null_counts[field.name] or 0),
        round(
            (null_counts[field.name] or 0)
            / row_count
            * 100,
            2
        ),
    )
    for field in df_raw.schema.fields
]

missing_values_df = spark.createDataFrame(
    missing_values_data,
    [
        "column_name",
        "data_type",
        "null_count",
        "null_percentage",
    ],
)

display(
    missing_values_df.orderBy(
        F.col("null_percentage").desc(),
        F.col("column_name"),
    )
)

column_name,data_type,null_count,null_percentage
CANCELLATION_CODE,string,6898743,98.53
CARRIER_DELAY,double,5466981,78.08
LATE_AIRCRAFT_DELAY,double,5466981,78.08
NAS_DELAY,double,5466981,78.08
SECURITY_DELAY,double,5466981,78.08
WEATHER_DELAY,double,5466981,78.08
ACTUAL_ELAPSED_TIME,double,122135,1.74
AIR_TIME,double,122135,1.74
ARR_DEL15,double,122135,1.74
ARR_DELAY,double,122135,1.74


#### Duplicate-record analysis

This section identifies fully duplicated rows that could distort flight counts and analytical results.

In [0]:
# Calculate fully duplicated records

distinct_record_count = df_raw.distinct().count()
duplicate_record_count = row_count - distinct_record_count

duplicate_percentage = round(
    duplicate_record_count / row_count * 100,
    4
)

print(f"Distinct records: {distinct_record_count:,}")
print(f"Duplicate records: {duplicate_record_count:,}")
print(f"Duplicate percentage: {duplicate_percentage}%")

Distinct records: 7,001,619
Duplicate records: 0
Duplicate percentage: 0.0%


#### Numerical summary

This section summarizes the distribution of numerical variables, including their central tendency, variation, ranges, and quartiles.

In [0]:
# Identify numerical columns

numeric_columns = [
    field.name
    for field in df_raw.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(f"Numerical columns: {len(numeric_columns)}")

Numerical columns: 23


In [0]:
# Display descriptive statistics for numerical columns

display(
    df_raw
    .select(numeric_columns)
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "max",
    )
)

summary,QUARTER,MONTH,DAY_OF_WEEK,OP_CARRIER_FL_NUM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
count,7001619,7001619,7001619,7001619,7001619,6903028,6903028,6899436,6897011,7001619,6879484,6879484,7001619,7001619,7001617,6879484,6879484,7001619,1534638,1534638,1534638,1534638,1534638
mean,2.5238985440367436,6.571155899799746,3.999110062972578,2517.2882784681656,1323.9459002267904,13.55640669572831,0.21756032280326837,18.64623238769082,8.611399053880007,1492.304588695843,8.504504407598011,0.2230745794306666,0.01469317310753413,0.0027505067042351205,148.9098054063797,144.08510042322942,116.84434065113022,844.4473882397771,23.667129968109744,4.632176448126529,15.95794317617575,0.09956875823484106,28.581965909875816
stddev,1.1053655450121083,3.3941441738648606,2.0105750716788204,1632.1071649054154,492.21964345817094,57.538035656825855,0.4125867828771151,10.69015175867992,7.35240474869369,518.5739720584324,59.735575643685124,0.41630798291064997,0.12032159340397561,0.05237310195952975,72.95441364646462,73.25642635293597,71.25512780390783,601.9752060669629,74.73646020023752,34.81616637943104,36.43684068395455,3.336252368528485,61.76801607785078
min,1,1,1,1,1,-115.0,0.0,1.0,1.0,1,-128.0,0.0,0.0,0.0,-99.0,15.0,7.0,31.0,0.0,0.0,0.0,0.0,0.0
25%,2,4,2,1197,904,-6.0,0.0,12.0,5.0,1103,-15.0,0.0,0.0,0.0,95.0,90.0,64.0,403.0,0.0,0.0,0.0,0.0,0.0
50%,3,7,4,2270,1318,-2.0,0.0,16.0,7.0,1518,-6.0,0.0,0.0,0.0,132.0,128.0,100.0,693.0,2.0,0.0,1.0,0.0,1.0
75%,4,10,6,3645,1735,10.0,0.0,21.0,10.0,1926,11.0,0.0,0.0,0.0,180.0,176.0,147.0,1080.0,20.0,0.0,19.0,0.0,33.0
max,4,12,7,9914,2400,4352.0,1.0,1274.0,1318.0,2400,4336.0,1.0,1.0,1.0,1510.0,965.0,953.0,5095.0,4336.0,2394.0,1706.0,990.0,2425.0


#### Key categorical variables

This section reviews the cardinality and most frequent values of the main categorical variables used in the analysis.

In [0]:
# Keep only configured categorical columns available in the dataset

categorical_columns = [
    column_name
    for column_name in cfg.KEY_CATEGORICAL_COLUMNS
    if column_name in df_raw.columns
]

categorical_summary = []

for column_name in categorical_columns:
    distinct_count = (
        df_raw
        .select(column_name)
        .distinct()
        .count()
    )

    categorical_summary.append(
        (column_name, distinct_count)
    )

categorical_summary_df = spark.createDataFrame(
    categorical_summary,
    [
        "column_name",
        "distinct_value_count",
    ],
)

display(
    categorical_summary_df.orderBy(
        F.col("distinct_value_count").desc()
    )
)

column_name,distinct_value_count
ORIGIN,352
DEST,352
OP_UNIQUE_CARRIER,14
CANCELLATION_CODE,5


In [0]:
# Display the most frequent values for key categorical columns

for column_name in categorical_columns:
    print(f"Most frequent values for {column_name}:")

    display(
        df_raw
        .groupBy(column_name)
        .count()
        .orderBy(F.col("count").desc())
        .limit(cfg.TOP_N_RESULTS)
    )

Most frequent values for OP_UNIQUE_CARRIER:


OP_UNIQUE_CARRIER,count
WN,1391885
DL,1026332
AA,973653
OO,839821
UA,795271
YX,346036
MQ,299322
OH,248735
AS,245588
B6,231413


Most frequent values for ORIGIN:


ORIGIN,count
ORD,327028
DEN,317686
DFW,315854
ATL,314433
PHX,196756
CLT,195425
LAX,190472
LAS,183803
SEA,164835
MCO,160048


Most frequent values for DEST:


DEST,count
ORD,327011
DEN,317675
DFW,315852
ATL,314439
PHX,196757
CLT,195428
LAX,190499
LAS,183787
SEA,164834
MCO,160000


Most frequent values for CANCELLATION_CODE:


CANCELLATION_CODE,count
null,6898743
B,63404
A,20903
C,18519
D,50


#### Essential data-quality checks

This section verifies the valid ranges and values of the main operational variables.

In [0]:
# Perform essential range and binary-value checks

quality_checks = [
    (
        "Invalid QUARTER values",
        df_raw.filter(
            F.col(cfg.QUARTER_COLUMN).isNull()
            | ~F.col(cfg.QUARTER_COLUMN).between(1, 4)
        ).count(),
    ),
    (
        "Invalid MONTH values",
        df_raw.filter(
            F.col(cfg.MONTH_COLUMN).isNull()
            | ~F.col(cfg.MONTH_COLUMN).between(1, 12)
        ).count(),
    ),
    (
        "Invalid DAY_OF_WEEK values",
        df_raw.filter(
            F.col(cfg.DAY_OF_WEEK_COLUMN).isNull()
            | ~F.col(cfg.DAY_OF_WEEK_COLUMN).between(1, 7)
        ).count(),
    ),
    (
        "Invalid CANCELLED values",
        df_raw.filter(
            F.col(cfg.CANCELLED_COLUMN).isNull()
            | ~F.col(cfg.CANCELLED_COLUMN).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid DIVERTED values",
        df_raw.filter(
            F.col(cfg.DIVERTED_COLUMN).isNull()
            | ~F.col(cfg.DIVERTED_COLUMN).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid ARR_DEL15 values",
        df_raw.filter(
            F.col(cfg.ARRIVAL_DELAY_FLAG_COLUMN).isNotNull()
            & ~F.col(
                cfg.ARRIVAL_DELAY_FLAG_COLUMN
            ).isin(0, 1)
        ).count(),
    ),
    (
        "Invalid DEP_DEL15 values",
        df_raw.filter(
            F.col(cfg.DEPARTURE_DELAY_FLAG_COLUMN).isNotNull()
            & ~F.col(
                cfg.DEPARTURE_DELAY_FLAG_COLUMN
            ).isin(0, 1)
        ).count(),
    ),
    (
        "Non-positive DISTANCE values",
        df_raw.filter(
            F.col(cfg.DISTANCE_COLUMN).isNull()
            | (F.col(cfg.DISTANCE_COLUMN) <= 0)
        ).count(),
    ),
]

quality_checks_df = spark.createDataFrame(
    quality_checks,
    [
        "quality_check",
        "invalid_record_count",
    ],
)

display(quality_checks_df)

quality_check,invalid_record_count
Invalid QUARTER values,0
Invalid MONTH values,0
Invalid DAY_OF_WEEK values,0
Invalid CANCELLED values,0
Invalid DIVERTED values,0
Invalid ARR_DEL15 values,0
Invalid DEP_DEL15 values,0
Non-positive DISTANCE values,0


#### Dataset preview

A small sample is displayed to support a final visual review of the raw records.

In [0]:
display(df_raw.limit(10))

QUARTER,MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_DELAY,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_DELAY,ARR_DEL15,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
3,7,1,7/7/2025 12:00:00 AM,AA,1,JFK,"New York, NY",New York,LAX,"Los Angeles, CA",California,720,-5.0,0.0,18.0,7.0,1022,-26.0,0.0,0.0,null,0.0,362.0,341.0,316.0,2475.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,10,LAX,"Los Angeles, CA",California,JFK,"New York, NY",New York,2121,4.0,0.0,22.0,11.0,559,-14.0,0.0,0.0,null,0.0,338.0,320.0,287.0,2475.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,1002,MSN,"Madison, WI",Wisconsin,CLT,"Charlotte, NC",North Carolina,716,-7.0,0.0,13.0,19.0,1030,-12.0,0.0,0.0,null,0.0,134.0,129.0,97.0,708.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,1003,CLT,"Charlotte, NC",North Carolina,MCI,"Kansas City, MO",Missouri,1615,1.0,0.0,71.0,6.0,1737,48.0,1.0,0.0,null,0.0,142.0,189.0,112.0,808.0,0.0,0.0,47.0,0.0,1.0
3,7,1,7/7/2025 12:00:00 AM,AA,1003,MCI,"Kansas City, MO",Missouri,CLT,"Charlotte, NC",North Carolina,1827,38.0,1.0,16.0,19.0,2155,29.0,1.0,0.0,null,0.0,148.0,139.0,104.0,808.0,0.0,0.0,0.0,0.0,29.0
3,7,1,7/7/2025 12:00:00 AM,AA,1004,BOS,"Boston, MA",Massachusetts,DCA,"Washington, DC",Virginia,2106,67.0,1.0,26.0,6.0,2250,63.0,1.0,0.0,null,0.0,104.0,100.0,68.0,399.0,0.0,0.0,0.0,0.0,63.0
3,7,1,7/7/2025 12:00:00 AM,AA,1007,CLT,"Charlotte, NC",North Carolina,PNS,"Pensacola, FL",Florida,2301,-5.0,0.0,33.0,6.0,2354,-10.0,0.0,0.0,null,0.0,113.0,108.0,69.0,488.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,1008,ATL,"Atlanta, GA",Georgia,DFW,"Dallas/Fort Worth, TX",Texas,1633,148.0,1.0,23.0,44.0,1801,180.0,1.0,0.0,null,0.0,148.0,180.0,113.0,731.0,0.0,0.0,32.0,0.0,148.0
3,7,1,7/7/2025 12:00:00 AM,AA,1009,LAX,"Los Angeles, CA",California,ORD,"Chicago, IL",Illinois,2259,-5.0,0.0,20.0,4.0,518,-33.0,0.0,0.0,null,0.0,259.0,231.0,207.0,1744.0,null,null,null,null,null
3,7,1,7/7/2025 12:00:00 AM,AA,1010,DFW,"Dallas/Fort Worth, TX",Texas,STL,"St. Louis, MO",Missouri,2105,6.0,0.0,15.0,6.0,2257,-9.0,0.0,0.0,null,0.0,112.0,97.0,76.0,550.0,null,null,null,null,null


#### Profiling completion

In [0]:
print("Data profiling completed successfully.")

Data profiling completed successfully.
